In [91]:
import os
import requests
import requests
import csv
from datetime import date, datetime, timedelta


REDCAP_API_TOKEN = 'B40BBD579D769085375A5179F942F093'

API_URL = "https://population.ahri.org/api/"


data = {
    'token': REDCAP_API_TOKEN,
    'content': 'project',
    'format': 'json',
    'returnFormat': 'json'
}
r = requests.post(API_URL, data=data)
print('HTTP Status: ' + str(r.status_code))
print(r.json())

HTTP Status: 200
{'project_id': 715, 'project_title': 'COVID-19 Mechanisms New Database', 'creation_time': '2021-10-20 14:32:25', 'production_time': '2022-04-25 14:06:36', 'in_production': 1, 'project_language': 'English', 'purpose': 2, 'purpose_other': '1,5', 'project_notes': '', 'custom_record_label': '', 'secondary_unique_field': '', 'is_longitudinal': 1, 'has_repeating_instruments_or_events': 1, 'surveys_enabled': 1, 'scheduling_enabled': 1, 'record_autonumbering_enabled': 0, 'randomization_enabled': 0, 'ddp_enabled': 0, 'project_irb_number': '', 'project_grant_number': '', 'project_pi_firstname': 'Alex', 'project_pi_lastname': 'Sigal', 'project_pi_email': 'alex.sigal@ahri.org', 'display_today_now_button': 1, 'missing_data_codes': '-999, Not previously asked | -555, Asked but unknown | NI, No information', 'external_modules': 'data_dictionary_revisions,instance_table,hide_submit', 'bypass_branching_erase_field_prompt': 0}


In [92]:
data = {
    'token': REDCAP_API_TOKEN,
    'content': 'record',
    'format': 'json',
    'returnFormat': 'json',
    'events': 'termination_arm_1'
}

r = requests.post(API_URL, data=data)

records = r.json()

print(records[:3])

[{'id_record': '039-02-0001', 'redcap_event_name': 'termination_arm_1', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': '', 'termination_date': '2023-09-04', 'termination_reason': '1', 'dod': '', 'specify_termtn_reason': '', 'termination_comments': '', 'termination_complete': '2', 'qr_pid_1': '', 'pid_qc_2': '', 'pid_qc_5': '', 'pid_qc_4': '', 'pid_qc_6': '', 'pid_qc_7': '', 'missed_visit_yn': '', 'missed_visit_date': '', 'missed_visit_reason': '', 'transfer_facility': '', 'specify_missed_vis_reason': '', 'missed_visits_complete': '', 'pc_visitdate': '', 'compby': '', 'pc_collector_site': '', 'fname': '', 'surnm': '', 'hid': '', 'pc_prep_personid': '', 'pc_update_contact_prim': '', 'second_telephone_number': '', 'pc_email': '', 'pc_physical_address': '', 'province': '', 'city_of_residence': '', 'commute_to_dbn_followup': '', 'specify_city_of_residence': '', 'nationality': '', 'dob': '', 'age': '', 'sex': '', 'race': '', 'over_18': '', 'pui': '', 'vaccine': '', 'positive_covid

In [93]:
records = r.json()

print(f"Arm: {arm_info['name']}")
print(f"Number of records: {len(records)}")

for i, rec in enumerate(records):
    try:
        if rec.get('termination_date') or rec.get('dod'):
            _ = rec['id_record']
    except KeyError:
        print(f"Problem at record {i}")
        print(rec)
        raise

Arm: Case
Number of records: 583


In [95]:

data = {
    'token': REDCAP_API_TOKEN,
    'content': 'arm',
    'format': 'json',
    'returnFormat': 'json'
}
r = requests.post(API_URL, data=data)
print(r.json())

[{'arm_num': 1, 'name': 'Case'}, {'arm_num': 2, 'name': 'Controls'}, {'arm_num': 3, 'name': 'Vaccinated'}, {'arm_num': 4, 'name': 'PUI'}, {'arm_num': 8, 'name': 'PAEDS'}]


In [96]:
data = {
    'token': REDCAP_API_TOKEN,
    'content': 'event',
    'format': 'json',
    'returnFormat': 'json'
}
r = requests.post(API_URL, data=data)
print(r.json())

[{'event_name': 'Termination', 'arm_num': 1, 'day_offset': -7, 'offset_min': 0, 'offset_max': 0, 'unique_event_name': 'termination_arm_1', 'custom_event_label': 'Termination', 'event_id': 3924}, {'event_name': 'Case/Ctrl - Day 1', 'arm_num': 1, 'day_offset': 0, 'offset_min': 0, 'offset_max': 0, 'unique_event_name': 'casectrl__day_1_arm_1', 'custom_event_label': 'Case/Control - Day 1', 'event_id': 3898}, {'event_name': 'Case/Ctrl - Day 7', 'arm_num': 1, 'day_offset': 7, 'offset_min': 0, 'offset_max': 0, 'unique_event_name': 'casectrl__day_7_arm_1', 'custom_event_label': 'Case/Control - Day 7', 'event_id': 3899}, {'event_name': 'Case/Ctrl - Day 14', 'arm_num': 1, 'day_offset': 14, 'offset_min': 0, 'offset_max': 0, 'unique_event_name': 'casectrl__day_14_arm_1', 'custom_event_label': 'Case/Control - Day 14', 'event_id': 3900}, {'event_name': 'Case/Ctrl - Day 21', 'arm_num': 1, 'day_offset': 21, 'offset_min': 0, 'offset_max': 0, 'unique_event_name': 'casectrl__day_21_arm_1', 'custom_event_l

In [97]:
import requests

# Map of each arm's number to its name and the two REDCap events we need:
# - enrollment_event: the Day 1 event where a participant first appears in this arm
# - termination_event: the event where withdrawal/death is recorded, if it happened
arms = {
    1: {'name': 'Case', 'termination_event': 'termination_arm_1', 'enrollment_event': 'casectrl__day_1_arm_1'},
    2: {'name': 'Controls', 'termination_event': 'termination_arm_2', 'enrollment_event': 'controls_day_1_arm_2'},
    3: {'name': 'Vaccinated', 'termination_event': 'termination_arm_3', 'enrollment_event': 'vaccinated_day_1_arm_3'},
    4: {'name': 'PUI', 'termination_event': 'termination_arm_4', 'enrollment_event': 'pui_day_1_arm_4'},
    8: {'name': 'PAEDS', 'termination_event': 'termination_arm_8', 'enrollment_event': 'paeds__day_1_arm_8'},
}


active_counts = {}          # per-arm counts, same as before (kept for reference)
all_active_ids = set()      # union of every active id_record across all arms, deduplicated

for arm_num, arm_info in arms.items():

    # --- Step 1: Get every participant enrolled in this arm ---
    data = {
        'token': REDCAP_API_TOKEN,
        'content': 'record',
        'format': 'json',
        'returnFormat': 'json',
        'fields': 'id_record',
        'events': arm_info['enrollment_event']
    }
    r = requests.post(API_URL, data=data)
    enrolled_ids = set(rec['id_record'] for rec in r.json() if rec.get('id_record'))

    # --- Step 2: Get every participant who has left this arm ---
    # A participant counts as terminated if EITHER termination_date OR dod
    # (date of death) has been recorded.
    data['fields'] = 'id_record,termination_date,dod'
    data['events'] = arm_info['termination_event']
    r = requests.post(API_URL, data=data)
    terminated_ids = set(
        rec['id_record'] for rec in r.json()
        if rec.get('termination_date') or rec.get('dod')
    )

    # --- Step 3: Active = enrolled minus terminated, for this arm ---
    active_ids = enrolled_ids - terminated_ids
    active_counts[arm_info['name']] = len(active_ids)

    # --- Step 4: Fold this arm's active ids into the running overall set ---
    # Using |= (set union) means a participant active in more than one arm
    # only ever gets added once — the union naturally deduplicates them.
    all_active_ids |= active_ids

# --- Print per-arm breakdown (may double-count participants who moved arms) ---
print("Active participants per arm:")
for arm_name, count in active_counts.items():
    print(f"  {arm_name}: {count}")
print(f"  Sum across arms (may include overlap): {sum(active_counts.values())}")

# --- Print the true, deduplicated total ---
print(f"\nTotal DISTINCT active participants across the whole cohort: {len(all_active_ids)}")

Active participants per arm:
  Case: 446
  Controls: 459
  Vaccinated: 624
  PUI: 341
  PAEDS: 338
  Sum across arms (may include overlap): 2208

Total DISTINCT active participants across the whole cohort: 2208


In [108]:


# Map of each arm's number to its name and the two REDCap events we need:
# - enrollment_event: the Day 1 event where a participant first appears in this arm
# - termination_event: the event where withdrawal/death is recorded, if it happened
arms = {
    1: {'name': 'Case', 'termination_event': 'termination_arm_1', 'enrollment_event': 'casectrl__day_1_arm_1'},
    2: {'name': 'Controls', 'termination_event': 'termination_arm_2', 'enrollment_event': 'controls_day_1_arm_2'},
    3: {'name': 'Vaccinated', 'termination_event': 'termination_arm_3', 'enrollment_event': 'vaccinated_day_1_arm_3'},
    4: {'name': 'PUI', 'termination_event': 'termination_arm_4', 'enrollment_event': 'pui_day_1_arm_4'},
    8: {'name': 'PAEDS', 'termination_event': 'termination_arm_8', 'enrollment_event': 'paeds__day_1_arm_8'},
}

# Anchor point for "upcoming" — anything due before today is excluded later
today = date.today()

# --- Step 1: Get every event defined in the project, with its arm and day offset ---
# day_offset tells us how many days after a participant's Day 1 visit each
# subsequent visit is expected (e.g. Day 7, Month 3, etc.)
data = {'token': REDCAP_API_TOKEN, 'content': 'event', 'format': 'json', 'returnFormat': 'json'}

r = requests.post(API_URL, data=data)

print(r.status_code)

records = r.json()

print(type(records))

if isinstance(records, dict):
    print(records)
else:
    print(records[:3])

events = r.json()

# Group events by arm and sort by offset, so we process them in visit order.
# Negative offsets (like Termination at -7) are excluded here since they're
# not real upcoming visits to schedule.
events_by_arm = {}
for arm_num in arms:
    events_by_arm[arm_num] = sorted(
        [e for e in events if e['arm_num'] == arm_num and e['day_offset'] >= 0],
        key=lambda e: e['day_offset']
    )

# Will hold one row per (arm, participant, event, due_date) for every
# upcoming visit across the whole cohort
schedule_rows = []

for arm_num, arm_info in arms.items():

    print(f"\n========== {arm_info['name']} ==========")
    

    # --- Step 2a: Get each participant's Day 1 (enrollment) visit date ---
    # This date is the "anchor" we use to calculate every later visit's due date.
    data = {
    'token': REDCAP_API_TOKEN,
    'content': 'record',
    'format': 'json',
    'returnFormat': 'json',
    'events': arm_info['enrollment_event']
}

r = requests.post(API_URL, data=data)

enrollment = {
    rec['id_record']: rec.get('pc_visitdate')
    for rec in r.json()
    if rec.get('id_record')
}


print(f"Enrollment records: {len(enrollment)}")

print(f"\n{arm_info['name']}")

print(f"Participants: {len(enrollment)}")

missing = sum(1 for d in enrollment.values() if not d)

print(f"Missing enrollment dates: {missing}")

print("First five:")

for i, (pid, dt) in enumerate(enrollment.items()):
    print(pid, dt)
    if i == 4:
        break


    # --- Step 2b: Get who has left this arm (terminated or deceased) ---
    # Reuses the same request dict, just swapping fields/events.
data = {
    'token': REDCAP_API_TOKEN,
    'content': 'record',
    'format': 'json',
    'returnFormat': 'json',
    'events': arm_info['termination_event']
}

r = requests.post(API_URL, data=data)

terminated = {
    rec['id_record']
    for rec in r.json()
    if rec.get('id_record')
    and (rec.get('termination_date') or rec.get('dod'))
}

print(f"Terminated: {len(terminated)}")

    # Active participants = enrolled minus terminated/deceased
active_ids = set(enrollment.keys()) - terminated

print(f"Active: {len(active_ids)}")

rows_before = len(schedule_rows)

    # --- Step 3: Calculate each active participant's upcoming visit dates ---
for pid in active_ids:
        anchor_str = enrollment.get(pid)
        if not anchor_str:
            continue  # skip participants with no recorded Day 1 visit date

        # Convert the enrollment date string into an actual date object
        try:
            anchor_date = datetime.strptime(anchor_str, '%Y-%m-%d').date()
        except ValueError:
            continue  # skip if the date is malformed/unparseable

        # For every event defined in this participant's arm, calculate the
        # actual calendar date it falls on (anchor date + day offset)
        
        for event in events_by_arm[arm_num]:
            due_date = anchor_date + timedelta(days=event['day_offset'])
            # if due_date >= today:  # only keep visits that haven't happened yet
            schedule_rows.append(
                (
                        arm_info["name"],
                        pid,
                        event["event_name"],
                        due_date.isoformat()
                )
)

rows_added = len(schedule_rows) - rows_before
print(f"Schedule rows added: {rows_added}")

print(f"Total upcoming scheduled visit rows: {len(schedule_rows)}")

# --- Step 4: Sort chronologically and write to CSV ---
# Sorting by due date (index 3 in each tuple) means the earliest upcoming
# visits appear first in the file.
schedule_rows.sort(key=lambda x: (x[0], x[1], x[3]))


print(f"Number of rows: {len(schedule_rows)}")

rows_before = len(schedule_rows)


for row in schedule_rows[:10]:
    print(row)


with open('mechanism_cohort_schedule_upcoming.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Arm', 'Participant_ID', 'Event', 'Due_Date'])
    writer.writerows(schedule_rows)

print("Upcoming schedule written to mechanism_cohort_schedule_upcoming.csv")

200
<class 'list'>
[{'event_name': 'Termination', 'arm_num': 1, 'day_offset': -7, 'offset_min': 0, 'offset_max': 0, 'unique_event_name': 'termination_arm_1', 'custom_event_label': 'Termination', 'event_id': 3924}, {'event_name': 'Case/Ctrl - Day 1', 'arm_num': 1, 'day_offset': 0, 'offset_min': 0, 'offset_max': 0, 'unique_event_name': 'casectrl__day_1_arm_1', 'custom_event_label': 'Case/Control - Day 1', 'event_id': 3898}, {'event_name': 'Case/Ctrl - Day 7', 'arm_num': 1, 'day_offset': 7, 'offset_min': 0, 'offset_max': 0, 'unique_event_name': 'casectrl__day_7_arm_1', 'custom_event_label': 'Case/Control - Day 7', 'event_id': 3899}]

========== Case ==========

========== Controls ==========

========== Vaccinated ==========

========== PUI ==========

========== PAEDS ==========
Enrollment records: 496

PAEDS
Participants: 496
Missing enrollment dates: 335
First five:
039-02-8001 2021-01-28
039-02-8002 2021-01-29
039-02-8003 2021-02-03
039-02-8004 2021-02-18
039-02-8005 2021-02-25
Termin